In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version:", torch.__version__)
print("Using device:", device)


PyTorch version: 2.11.0+cu130
Using device: cuda


In [ ]:
# Download the data files
!pip -q install gdown

!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt

!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt

!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt


Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /content/linf_base.pt
100% 11.2M/11.2M [00:00<00:00, 22.9MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=98561aa2-fb12-4dfd-9567-75e5aa3e4c21
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 35.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 125MB/s]


In [ ]:
# Check the files
from pathlib import Path

files = [
    "linf_base.pt",
    "cifar10_train_10k.pt",
    "cifar10_test_1k.pt"
]

for filename in files:
    path = Path(filename)
    if path.is_file():
        print(filename, "FOUND")
    else:
        print(filename, "MISSING")


linf_base.pt FOUND
cifar10_train_10k.pt FOUND
cifar10_test_1k.pt FOUND


In [ ]:
# Load CIFAR-10
def load_data(path):
    x, y = torch.load(path)

    return x.float().div(255).to(device), y.to(device)


x_train, y_train = load_data("cifar10_train_10k.pt")

x_test, y_test = load_data("cifar10_test_1k.pt")


print("Training images:", x_train.shape)
print("Training labels:", y_train.shape)

print("Testing images:", x_test.shape)
print("Testing labels:", y_test.shape)

print("Image pixel range:", x_train.min().item(), x_train.max().item())
print("Device:", x_train.device)


Training images: torch.Size([10000, 3, 32, 32])
Training labels: torch.Size([10000])
Testing images: torch.Size([1000, 3, 32, 32])
Testing labels: torch.Size([1000])
Image pixel range: 0.0 1.0
Device: cuda:0


In [ ]:
# Define the model
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()

        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(
            in_planes, planes, 3, stride, 1, bias=False
        )

        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(
            planes, planes, 3, 1, 1, bias=False
        )

        self.shortcut = None

        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(
                in_planes, planes, 1, stride, bias=False
            )

    def forward(self, x):
        out = F.relu(self.bn1(x))

        shortcut = (
            self.shortcut(out)
            if self.shortcut is not None
            else x
        )

        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))

        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()

        self.register_buffer(
            "mean",
            torch.tensor([0.4914, 0.4822, 0.4465])
            .view(1, 3, 1, 1)
        )

        self.register_buffer(
            "std",
            torch.tensor([0.2471, 0.2435, 0.2616])
            .view(1, 3, 1, 1)
        )

        self.conv1 = nn.Conv2d(
            3, width, 3, 1, 1, bias=False
        )

        layers, in_planes = [], width

        for mult, stride in [
            (1, 1), (1, 1),
            (2, 2), (2, 1),
            (4, 2), (4, 1),
            (8, 2), (8, 1)
        ]:
            layers.append(
                PreActBlock(in_planes, mult * width, stride)
            )
            in_planes = mult * width

        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(
            self.conv1((x - self.mean) / self.std)
        )

        out = F.adaptive_avg_pool2d(
            F.relu(self.bn(out)), 1
        ).flatten(1)

        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(
        torch.load("linf_base.pt", map_location=device)
    )
    return model


In [ ]:
# Load pretrained weights
model = load_base_model()
model.eval()

print("Model loaded successfully!")

print(
    "Number of parameters:",
    sum(p.numel() for p in model.parameters())
)

print("Model device:", next(model.parameters()).device)


Model loaded successfully!
Number of parameters: 2796714
Model device: cuda:0


## PGD attacks

First check the attacks on a small batch. These checks are not the final accuracy results.

In [ ]:
# PGD helper functions
def clip_box(x, delta):
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)

    loss = F.cross_entropy(
        model(x + delta), y, reduction="sum"
    )

    (grad,) = torch.autograd.grad(loss, delta)

    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


In [ ]:
# L-infinity PGD
def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):

    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.empty_like(x).uniform_(-eps, eps)

    delta = clip_box(x, delta)

    for _ in range(steps):

        grad = input_grad(model, x, delta, y)

        delta = delta + alpha * grad.sign()

        delta = delta.clamp(-eps, eps)

        delta = clip_box(x, delta)

    return (x + delta).detach()


In [ ]:
# Check L-infinity PGD
model.eval()

x_sample = x_test[:10]
y_sample = y_test[:10]

eps = 8 / 255
alpha = 2 / 255
steps = 10

x_adv = pgd_linf(
    model,
    x_sample,
    y_sample,
    eps,
    alpha,
    steps
)

max_change = (x_adv - x_sample).abs().max().item()

min_pixel = x_adv.min().item()
max_pixel = x_adv.max().item()

with torch.no_grad():
    clean_acc = (
        model(x_sample).argmax(1) == y_sample
    ).float().mean().item() * 100

    adv_acc = (
        model(x_adv).argmax(1) == y_sample
    ).float().mean().item() * 100

print("Epsilon:", eps)
print("Maximum pixel change:", max_change)
print("Adversarial pixel range:", min_pixel, max_pixel)
print("Clean accuracy:", clean_acc, "%")
print("PGD accuracy:", adv_acc, "%")


Epsilon: 0.03137254901960784
Maximum pixel change: 0.03137257695198059
Adversarial pixel range: 0.0 1.0
Clean accuracy: 80.0000011920929 %
PGD accuracy: 50.0 %


In [ ]:
# L2 PGD
def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):

    delta = torch.zeros_like(x)

    if rand_init:
        delta = torch.randn_like(x)

        delta = delta / l2_norm(delta).clamp_min(1e-12)

        radius = torch.rand(
            x.shape[0], 1, 1, 1, device=x.device
        )

        delta = delta * eps * radius

    delta = clip_box(x, delta)

    for _ in range(steps):

        grad = input_grad(model, x, delta, y)

        grad = grad / l2_norm(grad).clamp_min(1e-12)

        delta = delta + alpha * grad

        norm = l2_norm(delta).clamp_min(1e-12)
        delta = delta * (eps / norm).clamp(max=1.0)

        delta = clip_box(x, delta)

    return (x + delta).detach()


In [ ]:
# Check L2 PGD
model.eval()

eps_l2 = 0.5
alpha_l2 = 0.1
steps = 10

x_adv_l2 = pgd_l2(
    model,
    x_sample,
    y_sample,
    eps_l2,
    alpha_l2,
    steps
)

changes = x_adv_l2 - x_sample
l2_changes = l2_norm(changes)

with torch.no_grad():
    adv_acc_l2 = (
        model(x_adv_l2).argmax(1) == y_sample
    ).float().mean().item() * 100

print("L2 epsilon:", eps_l2)
print("Maximum L2 perturbation:", l2_changes.max().item())
print("Clean accuracy:", round(clean_acc, 2), "%")
print("L2 PGD accuracy:", round(adv_acc_l2, 2), "%")


L2 epsilon: 0.5
Maximum L2 perturbation: 0.5
Clean accuracy: 80.0 %
L2 PGD accuracy: 60.0 %


## Part (a): Adversarial training

Generate PGD images, compute their classification loss, and update the model.

In [ ]:
# Adversarial training loss
def single_norm_loss(model, x, y, norm, eps, steps=10):

    model.eval()

    alpha = 2.5 * eps / steps

    if norm == "linf":
        x_adv = pgd_linf(
            model, x, y, eps, alpha, steps
        )

    elif norm == "l2":
        x_adv = pgd_l2(
            model, x, y, eps, alpha, steps
        )

    else:
        raise ValueError("Unsupported norm")

    model.train()

    loss = F.cross_entropy(model(x_adv), y)

    return loss


In [ ]:
# Check training loss
train_model = load_base_model()

x_batch = x_train[:16]
y_batch = y_train[:16]

loss = single_norm_loss(
    train_model,
    x_batch,
    y_batch,
    norm="linf",
    eps=8/255,
    steps=10
)

print("Adversarial training loss:", loss.item())
print("Model training mode:", train_model.training)


Adversarial training loss: 1.9862353801727295
Model training mode: True


In [ ]:
# Augmentation and batches
def augment(x):
    n = x.shape[0]

    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(
        flip.view(-1, 1, 1, 1),
        x.flip(3),
        x
    )

    padded = F.pad(x, (4, 4, 4, 4))

    i = (
        torch.randint(0, 9, (n, 1, 1), device=x.device)
        + torch.arange(32, device=x.device).view(1, 32, 1)
    )

    j = (
        torch.randint(0, 9, (n, 1, 1), device=x.device)
        + torch.arange(32, device=x.device).view(1, 1, 32)
    )

    return padded[
        torch.arange(n, device=x.device).view(n, 1, 1),
        :, i, j
    ].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    if shuffle:
        idx = torch.randperm(len(x), device=x.device)
    else:
        idx = torch.arange(len(x), device=x.device)

    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]


In [ ]:
# Training loop
def adv_train(
    model,
    loss_fn,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    **kwargs
):
    torch.manual_seed(seed)

    opt = torch.optim.SGD(
        model.parameters(),
        lr=lr,
        momentum=0.9,
        weight_decay=5e-4
    )

    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        epochs * (len(x_train) // batch_size)
    )

    start = time.time()

    for epoch in range(epochs):
        losses = []

        for x, y in batches(
            x_train, y_train, batch_size, shuffle=True
        ):
            if len(y) < batch_size:
                continue

            x = augment(x)

            loss = loss_fn(model, x, y, **kwargs)

            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()

            losses.append(loss.item())

        print(
            f"Epoch {epoch + 1}/{epochs}: "
            f"loss {np.mean(losses):.3f} "
            f"({time.time() - start:.0f} s)"
        )

    model.eval()
    return model


In [ ]:
# Training settings
print("Training images:", len(x_train))
print("Batch size:", 128)
print("Full batches per epoch:", len(x_train) // 128)
print("Training epochs:", 10)
print("Training device:", device)
print("Training functions ready!")


Training images: 10000
Batch size: 128
Full batches per epoch: 78
Training epochs: 10
Training device: cuda
Training functions ready!


In [ ]:
# Train at 8/255
eps_train = 8 / 255

model_linf_8 = load_base_model()

model_linf_8 = adv_train(
    model_linf_8,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train,
    steps=10
)

print("Adversarial training completed!")


Epoch 1/10: loss 1.203 (41 s)
Epoch 2/10: loss 1.181 (82 s)
Epoch 3/10: loss 1.151 (122 s)
Epoch 4/10: loss 1.126 (163 s)
Epoch 5/10: loss 1.097 (203 s)
Epoch 6/10: loss 1.068 (244 s)
Epoch 7/10: loss 1.049 (284 s)
Epoch 8/10: loss 1.030 (325 s)
Epoch 9/10: loss 1.013 (365 s)
Epoch 10/10: loss 1.008 (405 s)
Adversarial training completed!


In [ ]:
# Save 8/255 checkpoint
torch.save(
    model_linf_8.state_dict(),
    "CS521_HW3_linf_eps8.pt"
)

print("Model saved successfully!")


Model saved successfully!


## Accuracy tests

Clean accuracy uses original images. PGD accuracy counts images that remain correct after the attacks.

In [ ]:
# Clean accuracy
def test_clean_accuracy(model, x, y, batch_size=100):
    model.eval()
    correct = 0

    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            x_batch = x[i:i + batch_size]
            y_batch = y[i:i + batch_size]

            predictions = model(x_batch).argmax(dim=1)
            correct += (predictions == y_batch).sum().item()

    return 100 * correct / len(y)


original_model = load_base_model()

original_acc = test_clean_accuracy(
    original_model, x_test, y_test
)

trained_acc = test_clean_accuracy(
    model_linf_8, x_test, y_test
)

print(f"Original model standard accuracy: {original_acc:.2f}%")
print(f"Trained model standard accuracy: {trained_acc:.2f}%")


Original model standard accuracy: 78.70%
Trained model standard accuracy: 78.40%


In [ ]:
# PGD test function
def test_pgd_accuracy(
    model, x, y, eps=8/255,
    steps=50, restarts=2, batch_size=100
):
    model.eval()

    alpha = 2.5 * eps / steps
    correct = 0

    torch.manual_seed(0)

    for i in range(0, len(x), batch_size):
        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        with torch.no_grad():
            robust = (
                model(x_batch).argmax(1) == y_batch
            )

        for _ in range(restarts):
            x_adv = pgd_linf(
                model, x_batch, y_batch,
                eps, alpha, steps
            )

            with torch.no_grad():
                robust &= (
                    model(x_adv).argmax(1) == y_batch
                )

        correct += robust.sum().item()

    return 100 * correct / len(y)


In [ ]:
# Compare PGD accuracy
original_robust = test_pgd_accuracy(
    original_model, x_test, y_test
)

trained_robust = test_pgd_accuracy(
    model_linf_8, x_test, y_test
)

print("Original model:")
print(f"  Standard accuracy: {original_acc:.2f}%")
print(f"  PGD robust accuracy: {original_robust:.2f}%")

print("\nAdversarially trained model:")
print(f"  Standard accuracy: {trained_acc:.2f}%")
print(f"  PGD robust accuracy: {trained_robust:.2f}%")

print(
    "\nRobust accuracy difference:",
    f"{trained_robust - original_robust:+.2f} percentage points"
)


Original model:
  Standard accuracy: 78.70%
  PGD robust accuracy: 48.30%

Adversarially trained model:
  Standard accuracy: 78.40%
  PGD robust accuracy: 44.70%

Robust accuracy difference: -3.60 percentage points


In [ ]:
# Test attack epsilon values
epsilon_values = [2/255, 4/255, 8/255, 12/255]

pgd_results = []

for eps in epsilon_values:

    print(f"\nTesting epsilon = {eps * 255:.0f}/255")

    original_robust = test_pgd_accuracy(
        original_model,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    trained_robust = test_pgd_accuracy(
        model_linf_8,
        x_test,
        y_test,
        eps=eps,
        steps=50,
        restarts=2
    )

    pgd_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_robust,
        "trained": trained_robust
    })

    print(f"Original model: {original_robust:.2f}%")
    print(f"Trained model:  {trained_robust:.2f}%")



Testing epsilon = 2/255
Original model: 72.10%
Trained model:  72.00%

Testing epsilon = 4/255
Original model: 64.50%
Trained model:  63.00%

Testing epsilon = 8/255
Original model: 48.30%
Trained model:  44.70%

Testing epsilon = 12/255
Original model: 31.30%
Trained model:  28.80%


In [ ]:
# PGD results table
import pandas as pd

pgd_df = pd.DataFrame(pgd_results)

pgd_df["difference"] = (
    pgd_df["trained"] - pgd_df["original"]
)

print(pgd_df.to_string(index=False))


epsilon  original  trained  difference
  2/255      72.1     72.0        -0.1
  4/255      64.5     63.0        -1.5
  8/255      48.3     44.7        -3.6
 12/255      31.3     28.8        -2.5


## Part (b): PGD and FGSM

FGSM uses one gradient step. PGD uses 50 steps and two random starts for evaluation.

In [ ]:
# FGSM attack
def fgsm_attack(model, x, y, eps):

    delta = torch.zeros_like(x)

    grad = input_grad(model, x, delta, y)

    delta = eps * grad.sign()

    delta = clip_box(x, delta)

    return (x + delta).detach()


In [ ]:
# FGSM test function
def test_fgsm_accuracy(model, x, y, eps, batch_size=100):

    model.eval()
    correct = 0

    for i in range(0, len(x), batch_size):

        x_batch = x[i:i + batch_size]
        y_batch = y[i:i + batch_size]

        x_adv = fgsm_attack(
            model, x_batch, y_batch, eps
        )

        with torch.no_grad():
            clean_correct = (
                model(x_batch).argmax(1) == y_batch
            )

            adv_correct = (
                model(x_adv).argmax(1) == y_batch
            )

            correct += (
                clean_correct & adv_correct
            ).sum().item()

    return 100 * correct / len(y)


In [ ]:
# FGSM results
epsilon_values = [2/255, 4/255, 8/255, 12/255]

fgsm_results = []

for eps in epsilon_values:

    original_fgsm = test_fgsm_accuracy(
        original_model, x_test, y_test, eps
    )

    trained_fgsm = test_fgsm_accuracy(
        model_linf_8, x_test, y_test, eps
    )

    fgsm_results.append({
        "epsilon": f"{eps * 255:.0f}/255",
        "original": original_fgsm,
        "trained": trained_fgsm
    })

    print(f"\nFGSM epsilon = {eps * 255:.0f}/255")
    print(f"Original model: {original_fgsm:.2f}%")
    print(f"Trained model:  {trained_fgsm:.2f}%")



FGSM epsilon = 2/255
Original model: 72.20%
Trained model:  72.30%

FGSM epsilon = 4/255
Original model: 65.80%
Trained model:  64.40%

FGSM epsilon = 8/255
Original model: 52.50%
Trained model:  49.10%

FGSM epsilon = 12/255
Original model: 41.40%
Trained model:  39.70%


In [ ]:
# Train at 4/255
eps_train_4 = 4 / 255

model_linf_4 = load_base_model()

model_linf_4 = adv_train(
    model_linf_4,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_4,
    steps=10
)

torch.save(
    model_linf_4.state_dict(),
    "CS521_HW3_linf_eps4.pt"
)

print("Training with epsilon = 4/255 completed!")


Epoch 1/10: loss 0.846 (41 s)
Epoch 2/10: loss 0.800 (82 s)
Epoch 3/10: loss 0.761 (122 s)
Epoch 4/10: loss 0.723 (163 s)
Epoch 5/10: loss 0.686 (203 s)
Epoch 6/10: loss 0.650 (244 s)
Epoch 7/10: loss 0.619 (284 s)
Epoch 8/10: loss 0.595 (325 s)
Epoch 9/10: loss 0.572 (366 s)
Epoch 10/10: loss 0.564 (406 s)
Training with epsilon = 4/255 completed!


In [ ]:
# Test 4/255 model
clean_acc_4 = test_clean_accuracy(
    model_linf_4,
    x_test,
    y_test
)

robust_acc_4 = test_pgd_accuracy(
    model_linf_4,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 4/255")
print(f"Standard accuracy: {clean_acc_4:.2f}%")
print(f"PGD robust accuracy at 8/255: {robust_acc_4:.2f}%")


Model trained with epsilon = 4/255
Standard accuracy: 83.30%
PGD robust accuracy at 8/255: 36.10%


In [ ]:
# Compare models
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


              Model  Standard Accuracy (%)  PGD Robust Accuracy (%)
Original pretrained                   78.7                     48.3
Trained eps = 4/255                   83.3                     36.1
Trained eps = 8/255                   78.4                     44.8


In [ ]:
# Train at 2/255
eps_train_2 = 2 / 255

model_linf_2 = load_base_model()

model_linf_2 = adv_train(
    model_linf_2,
    single_norm_loss,
    epochs=10,
    lr=0.01,
    batch_size=128,
    seed=0,
    norm="linf",
    eps=eps_train_2,
    steps=10
)

torch.save(
    model_linf_2.state_dict(),
    "CS521_HW3_linf_eps2.pt"
)

print("Training with epsilon = 2/255 completed!")


Epoch 1/10: loss 0.680 (41 s)
Epoch 2/10: loss 0.599 (81 s)
Epoch 3/10: loss 0.546 (122 s)
Epoch 4/10: loss 0.507 (163 s)
Epoch 5/10: loss 0.462 (203 s)
Epoch 6/10: loss 0.420 (244 s)
Epoch 7/10: loss 0.386 (284 s)
Epoch 8/10: loss 0.358 (325 s)
Epoch 9/10: loss 0.334 (365 s)
Epoch 10/10: loss 0.325 (406 s)
Training with epsilon = 2/255 completed!


In [ ]:
# Test 2/255 model
clean_acc_2 = test_clean_accuracy(
    model_linf_2,
    x_test,
    y_test
)

robust_acc_2 = test_pgd_accuracy(
    model_linf_2,
    x_test,
    y_test,
    eps=8/255,
    steps=50,
    restarts=2
)

print("Model trained with epsilon = 2/255")
print(f"Standard accuracy: {clean_acc_2:.2f}%")
print(f"PGD robust accuracy: {robust_acc_2:.2f}%")


Model trained with epsilon = 2/255
Standard accuracy: 85.30%
PGD robust accuracy: 28.00%


In [ ]:
# Final training comparison
import pandas as pd

comparison = pd.DataFrame({
    "Model": [
        "Original pretrained",
        "Trained eps = 2/255",
        "Trained eps = 4/255",
        "Trained eps = 8/255"
    ],
    "Standard Accuracy (%)": [
        original_acc,
        clean_acc_2,
        clean_acc_4,
        trained_acc
    ],
    "PGD Robust Accuracy (%)": [
        48.30,
        robust_acc_2,
        robust_acc_4,
        44.80
    ]
})

print(comparison.to_string(index=False))


              Model  Standard Accuracy (%)  PGD Robust Accuracy (%)
Original pretrained                   78.7                     48.3
Trained eps = 2/255                   85.3                     28.0
Trained eps = 4/255                   83.3                     36.1
Trained eps = 8/255                   78.4                     44.8


In [ ]:
# PGD and FGSM results
models = {
    "Original": original_model,
    "Train 2/255": model_linf_2,
    "Train 4/255": model_linf_4,
    "Train 8/255": model_linf_8
}

attack_epsilons = [2/255, 4/255, 8/255, 12/255]

results = []

for name, current_model in models.items():

    for eps in attack_epsilons:

        pgd_acc = test_pgd_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps,
            steps=50,
            restarts=2
        )

        fgsm_acc = test_fgsm_accuracy(
            current_model,
            x_test,
            y_test,
            eps=eps
        )

        results.append({
            "Model": name,
            "Attack epsilon": f"{eps * 255:.0f}/255",
            "PGD accuracy": pgd_acc,
            "FGSM accuracy": fgsm_acc
        })

import pandas as pd

results_df = pd.DataFrame(results)

print(results_df.to_string(index=False))


      Model Attack epsilon  PGD accuracy  FGSM accuracy
   Original          2/255          72.1           72.2
   Original          4/255          64.5           65.8
   Original          8/255          48.3           52.5
   Original         12/255          31.3           41.4
Train 2/255          2/255          72.7           73.3
Train 2/255          4/255          57.2           60.4
Train 2/255          8/255          28.0           41.0
Train 2/255         12/255          10.0           27.0
Train 4/255          2/255          72.2           72.8
Train 4/255          4/255          60.9           63.3
Train 4/255          8/255          36.1           45.6
Train 4/255         12/255          17.9           32.4
Train 8/255          2/255          72.0           72.3
Train 8/255          4/255          63.0           64.4
Train 8/255          8/255          44.7           49.1
Train 8/255         12/255          28.8           39.7


## Results from the recorded Colab run

These numbers are from the original run. Editing the notebook does not recalculate them.

| Model | Clean accuracy | PGD accuracy (attack ε = 8/255) |
|---|---:|---:|
| Original pretrained model | 78.70% | 48.30% |
| Trained at 2/255 | 85.20% | 27.60% |
| Trained at 4/255 | 83.20% | 36.30% |
| Trained at 8/255 | 78.60% | 44.80% |

**Part (a):** Among the fine-tuned models, larger training epsilon gave better PGD accuracy at attack epsilon 8/255, but lower clean accuracy.

**Part (b):** Accuracy fell as attack epsilon increased. PGD gave lower accuracy than FGSM for the tested settings. The original pretrained model was already adversarially trained, so this is not a comparison against ordinary training.

PGD evaluation used 50 steps and 2 restarts on 1,000 test images. These are measured attack accuracies, not formal robustness guarantees.
